In [39]:
import pandas as pd
import re
import numpy as np
pd.set_option('display.max_columns', None)

In [40]:
gt_ip = pd.read_csv('../data/raw/ehn_gt_inpatient_20251223000.csv')
gt_op = pd.read_csv('../data/raw/ehn_gt_outpatient_20251223000.csv')
risks = pd.read_csv('../data/raw/2024 Totals and Risk.csv')

/var/folders/v8/s4lwxw8s6wxbkhy0gqkmypnc0000gn/T/ipykernel_48647/3210601031.py:3: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  risks = pd.read_csv('../data/raw/2024 Totals and Risk.csv')


In [41]:
risks.head()

,deid_personid,membermonths,payertype,city,state,zip,pcpnpi,pcpname,ehn_vs_non,employed_vs_independent,ehn_practice,lhn,entity,riskhcctotalrisk,riskhccclosed,chronicconditionasthmaind,chronicconditionesrdind,chronicconditionhfind,chronicconditioncrfind,chronicconditionhypertensionind,chronicconditioncancerind,chronicconditioncopdind,chronicconditiondiabetesind,costtotal,countip,costip,ipreadmit,followup7daypcp,countop,costop,counted,costed,countavoidableed,countretailrx,costretailrx,lastvisitdate,lastvisitfacilityname
0,e90f424239154322af8f015e1f88bf69,2.0,MA,ELLENWOOD,GA,30294,1376625053,"ARMSTRONG, CHANDRA",EHN,EHN Independent,"Chandra Britt Armstrong MD, LLC dba Dekalb Fam...",EDH,PRIVATE,3.017,3.017,0,0,1,1,1,0,0,1,$(1.02),0,$-,0,0,0,$-,0,$-,0,4,$(1.02),NaN,NaN
1,649c4ca9a7d14e26361c3cb41e6d871d,5.0,Comm,CUMMING,GA,30041,Unknown,UNKNOWN,Unknown,Unknown,Unknown,Unknown,Unknown,0.224,0.224,0,0,0,0,0,0,0,0,$-,0,$-,0,0,10,$-,0,$-,0,0,$-,12/3/2024,NaN
2,db041b79ea79d30fd6ef69802f3ff818,12.0,Comm,AUSTELL,GA,30168,1053428540,"NWOSU, OGUCHI",EHN,EHN Employed,Emory at Dunwoody - Family Medicine,EUH,EC,0.122,0.122,0,0,0,0,0,0,0,0,$-,0,$-,0,0,3,$-,0,$-,0,0,$-,3/26/2024,NaN
3,932164ff1a357d3df333b56e67b77fa0,3.0,Comm,ATLANTA,GA,30340,1215011663,"CARR, ROGER",EHN,EHN Independent,"Southeast Medical Group, Fayetteville",SC,PRIVATE,0.285,0.285,1,0,0,0,0,0,0,0,$-,0,$-,0,0,0,$-,0,$-,0,0,$-,NaN,NaN
4,48858fec3d5287d5ba456465079f2a32,12.0,MA,LITHONIA,GA,30058,1598897530,"WHITE-WILLIAMS, DOROTHY",EHN,EHN Independent,"Greater Atlanta Family Healthcare, LLC",EDH,PRIVATE,0.324,0.324,0,0,0,0,0,0,0,0,$-,0,$-,0,0,0,$-,0,$-,0,0,$-,NaN,NaN


In [42]:
risks.columns

Index(['deid_personid', 'membermonths', 'payertype', 'city', 'state', 'zip',
       'pcpnpi', 'pcpname', 'ehn_vs_non', 'employed_vs_independent',
       'ehn_practice', 'lhn', 'entity', 'riskhcctotalrisk', 'riskhccclosed',
       'chronicconditionasthmaind', 'chronicconditionesrdind',
       'chronicconditionhfind', 'chronicconditioncrfind',
       'chronicconditionhypertensionind', 'chronicconditioncancerind',
       'chronicconditioncopdind', 'chronicconditiondiabetesind', ' costtotal ',
       'countip', ' costip ', 'ipreadmit', 'followup7daypcp', 'countop',
       ' costop ', 'counted', ' costed ', 'countavoidableed', 'countretailrx',
       ' costretailrx ', 'lastvisitdate', 'lastvisitfacilityname'],
      dtype='object')

#### Clean all cost columns

In [43]:
for name in ('risks', 'gt_ip', 'gt_op'):
    if name in globals():
        df = globals()[name]
        for col in df.columns:
            if str(col).strip().lower().startswith('cost'):
                s = df[col].astype(str).str.strip()
                # treat explicit "$-" (or variants) as zero
                s = s.replace(r'^\$[\s-]*$', '0', regex=True)
                # detect parentheses indicating negative amounts like $(1.02) or (1.02)
                neg = s.str.match(r'^\(.*\)$')
                # remove dollar, commas, parentheses
                s = s.str.replace(r'[\$\(\),]', '', regex=True).str.strip()
                # empty strings -> NaN
                s = s.replace(r'^\s*$', np.nan, regex=True)
                df[col] = pd.to_numeric(s, errors='coerce')
                if neg.any():
                    df.loc[neg, col] = -df.loc[neg, col]

#### Rename some columns in risks

In [44]:
cols_to_fix = [' costtotal ', ' costip ', ' costop ', ' costed ', ' costretailrx ']
rename_map = {c: c.strip() for c in cols_to_fix if c in risks.columns}
risks = risks.rename(columns=rename_map)

#### Make sure some column types are correct

In [45]:
chronicconditions_cols = [col for col in risks.columns if col.lower().startswith('chroniccondition')]
for col in chronicconditions_cols:
    risks[col] = pd.to_numeric(risks[col], errors='coerce')

count_cols = [col for col in risks.columns if col.lower().startswith('count')]
for col in count_cols:
    risks[col] = pd.to_numeric(risks[col], errors='coerce')
risks.head()

# make sure membermonths is numeric and non-zero to avoid division issues, and ranges from 1 to 12 since it's a 1-year period
risks['membermonths'] = pd.to_numeric(risks['membermonths'], errors='coerce')
risks = risks[(risks['membermonths'] >= 1) & (risks['membermonths'] <= 12)]

#### Remove rows where 'riskhcctotalrisk', 'riskhccclosed' have NaN  

In [46]:
# risks = risks.dropna(subset=['riskhcctotalrisk', 'riskhccclosed'])

#### In OP/IP data, keep only members who appear in risks data

In [47]:
# keep members who appeat in risks 

valid_memberids = set(risks['deid_personid'].dropna().unique())
gt_ip = gt_ip[gt_ip['deid_personid'].isin(valid_memberids)]
gt_op = gt_op[gt_op['deid_personid'].isin(valid_memberids)]

#### Save data to processed

In [48]:
from pathlib import Path

out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

gt_ip.to_csv(out_dir / "ehn_gt_inpatient_20251223000.csv", index=False)
gt_op.to_csv(out_dir / "ehn_gt_outpatient_20251223000.csv", index=False)
risks.to_csv(out_dir / "2024 Totals and Risk.csv", index=False)